--- Day 14: Parabolic Reflector Dish ---
You reach the place where all of the mirrors were pointing: a massive parabolic reflector dish attached to the side of another large mountain.

The dish is made up of many small mirrors, but while the mirrors themselves are roughly in the shape of a parabolic reflector dish, each individual mirror seems to be pointing in slightly the wrong direction. If the dish is meant to focus light, all it's doing right now is sending it in a vague direction.

This system must be what provides the energy for the lava! If you focus the reflector dish, maybe you can go where it's pointing and use the light to fix the lava production.

Upon closer inspection, the individual mirrors each appear to be connected via an elaborate system of ropes and pulleys to a large metal platform below the dish. The platform is covered in large rocks of various shapes. Depending on their position, the weight of the rocks deforms the platform, and the shape of the platform controls which ropes move and ultimately the focus of the dish.

In short: if you move the rocks, you can focus the dish. The platform even has a control panel on the side that lets you tilt it in one of four directions! The rounded rocks (O) will roll when the platform is tilted, while the cube-shaped rocks (#) will stay in place. You note the positions of all of the empty spaces (.) and rocks (your puzzle input). For example:

O....#....
O.OO#....#
.....##...
OO.#O....O
.O.....O#.
O.#..O.#.#
..O..#O..O
.......O..
#....###..
#OO..#....
Start by tilting the lever so all of the rocks will slide north as far as they will go:

OOOO.#.O..
OO..#....#
OO..O##..O
O..#.OO...
........#.
..#....#.#
..O..#.O.O
..O.......
#....###..
#....#....
You notice that the support beams along the north side of the platform are damaged; to ensure the platform doesn't collapse, you should calculate the total load on the north support beams.

The amount of load caused by a single rounded rock (O) is equal to the number of rows from the rock to the south edge of the platform, including the row the rock is on. (Cube-shaped rocks (#) don't contribute to load.) So, the amount of load caused by each rock in each row is as follows:

OOOO.#.O.. 10
OO..#....#  9
OO..O##..O  8
O..#.OO...  7
........#.  6
..#....#.#  5
..O..#.O.O  4
..O.......  3
#....###..  2
#....#....  1
The total load is the sum of the load caused by all of the rounded rocks. In this example, the total load is 136.

Tilt the platform so that the rounded rocks all roll north. Afterward, what is the total load on the north support beams?

In [1]:
# Strategy: Get all positions of the unmovable #-rocks
# Then adjoin all movable rock to the closest northern rock from them
# Then we can count: For each #, the attached weights is the number of rocks decremented by 1 for each additional rock

with open('input14.txt') as file:
    data = [l.rstrip() for l in file]
data[0]

'#..O.#..#...##O..O.......O.#..#.O#.#.....#O.#.O..#OO....O..O..O.OO....O.O#.O..........#....OOO..O##O'

In [2]:
from collections import Counter
s_rocks = {col : Counter({-1:0}) for col, _ in enumerate(data[0])}
for rx, l in enumerate(data):
    for cx, c in enumerate(l):
        if c == '#':
            s_rocks[cx][rx] = 0

for rx, l in enumerate(data):
    for cx, c in enumerate(l):
        if c == 'O':
            for i in range(rx - 1, -2, -1):
                if i in s_rocks[cx]:
                    s_rocks[cx][i] += 1
                    break
s_rocks[0]

Counter({77: 7,
         23: 3,
         66: 3,
         0: 2,
         17: 2,
         40: 2,
         14: 1,
         36: 1,
         46: 1,
         56: 1,
         59: 1,
         -1: 0,
         22: 0})

In [3]:
# tallying:
total = 0
for col in s_rocks.values():
    for pos, cnt in col.items():
        tally = sum(range(len(data) - pos - cnt, len(data) - pos))
        total += tally
total

109345

Your puzzle answer was 109345.

The first half of this puzzle is complete! It provides one gold star: *

--- Part Two ---
The parabolic reflector dish deforms, but not in a way that focuses the beam. To do that, you'll need to move the rocks to the edges of the platform. Fortunately, a button on the side of the control panel labeled "spin cycle" attempts to do just that!

Each cycle tilts the platform four times so that the rounded rocks roll north, then west, then south, then east. After each tilt, the rounded rocks roll as far as they can before the platform tilts in the next direction. After one cycle, the platform will have finished rolling the rounded rocks in those four directions in that order.

Here's what happens in the example above after each of the first few cycles:

After 1 cycle:
.....#....
....#...O#
...OO##...
.OO#......
.....OOO#.
.O#...O#.#
....O#....
......OOOO
#...O###..
#..OO#....

After 2 cycles:
.....#....
....#...O#
.....##...
..O#......
.....OOO#.
.O#...O#.#
....O#...O
.......OOO
#..OO###..
#.OOO#...O

After 3 cycles:
.....#....
....#...O#
.....##...
..O#......
.....OOO#.
.O#...O#.#
....O#...O
.......OOO
#...O###.O
#.OOO#...O
This process should work if you leave it running long enough, but you're still worried about the north support beams. To make sure they'll survive for a while, you need to calculate the total load on the north support beams after 1000000000 cycles.

In the above example, after 1000000000 cycles, the total load on the north support beams is 64.

Run the spin cycle for 1000000000 cycles. Afterward, what is the total load on the north support beams?

In [4]:
# Let us assume some kind of periodicity for the layout. We can check for repeats after each cycle to find out when the pattern repeats

stops_v = {col_ix : set((-1, len(data))) for col_ix, _ in enumerate(data)}
for rx, l in enumerate(data):
    for cx, c in enumerate(l):
        if c == '#':
            stops_v[cx].add(rx)

stops_h = {rx : set((-1, len(data[0]))) for rx, _ in enumerate(data)}
for rx, l in enumerate(data):
    for cx, c in enumerate(l):
        if c == '#':
            stops_h[rx].add(cx)
stops_h[0], stops_v[0]

({-1, 0, 5, 8, 12, 13, 27, 30, 33, 35, 41, 44, 49, 73, 86, 97, 98, 100},
 {-1, 0, 14, 17, 22, 23, 36, 40, 46, 56, 59, 66, 77, 100})

In [33]:
stones_init = []
for rx, l in enumerate(data):
    for cx, c in enumerate(l):
        if c == 'O':
            stones_init.append((rx, cx))
stones_init

[(0, 3),
 (0, 14),
 (0, 17),
 (0, 25),
 (0, 32),
 (0, 42),
 (0, 46),
 (0, 50),
 (0, 51),
 (0, 56),
 (0, 59),
 (0, 62),
 (0, 64),
 (0, 65),
 (0, 70),
 (0, 72),
 (0, 75),
 (0, 91),
 (0, 92),
 (0, 93),
 (0, 96),
 (0, 99),
 (1, 2),
 (1, 5),
 (1, 7),
 (1, 11),
 (1, 12),
 (1, 18),
 (1, 27),
 (1, 28),
 (1, 30),
 (1, 40),
 (1, 47),
 (1, 48),
 (1, 55),
 (1, 60),
 (1, 63),
 (1, 81),
 (1, 84),
 (1, 85),
 (1, 88),
 (1, 90),
 (1, 94),
 (1, 97),
 (2, 3),
 (2, 9),
 (2, 10),
 (2, 12),
 (2, 16),
 (2, 31),
 (2, 33),
 (2, 36),
 (2, 40),
 (2, 42),
 (2, 51),
 (2, 56),
 (2, 59),
 (2, 64),
 (2, 71),
 (2, 72),
 (2, 74),
 (2, 75),
 (2, 77),
 (2, 87),
 (2, 88),
 (2, 92),
 (3, 1),
 (3, 4),
 (3, 5),
 (3, 7),
 (3, 9),
 (3, 19),
 (3, 21),
 (3, 22),
 (3, 28),
 (3, 33),
 (3, 35),
 (3, 40),
 (3, 50),
 (3, 55),
 (3, 57),
 (3, 59),
 (3, 72),
 (3, 81),
 (3, 96),
 (4, 7),
 (4, 13),
 (4, 16),
 (4, 18),
 (4, 19),
 (4, 27),
 (4, 33),
 (4, 35),
 (4, 41),
 (4, 42),
 (4, 43),
 (4, 49),
 (4, 50),
 (4, 52),
 (4, 54),
 (4, 56),
 (

In [11]:
def tilt(stones, orient):
    def snap(stops, val, minus):
        if minus:
            for i in range(val - 1, -2, -1):
                if i in stops:
                    return i
        else:
            for i in range(val + 1, len(data) + 1):
                if i in stops:
                    return i
    
    out_count = {k : Counter() for k in range(len(data))}
    stops = stops_v if orient == 'north' or orient == 'south' else stops_h

    for rx, cx in stones:
        if orient == 'north':
            out_count[cx][snap(stops[cx], rx, minus=True)] += 1
        elif orient == 'south':
            out_count[cx][snap(stops[cx], rx, minus=False)] += 1
        elif orient == 'west':
            out_count[rx][snap(stops[rx], cx, minus=True)] += 1
        else:
            out_count[rx][snap(stops[rx], cx, minus=False)] += 1
    return out_count
a = tilt(stones, 'north')
a

{0: Counter({77: 7,
          66: 3,
          23: 3,
          0: 2,
          17: 2,
          40: 2,
          36: 1,
          56: 1,
          14: 1,
          46: 1,
          59: 1}),
 1: Counter({55: 3, 15: 3, 93: 3, 35: 2, 51: 2, 66: 1, 1: 1, 31: 1, 5: 1}),
 2: Counter({33: 3,
          62: 3,
          6: 2,
          13: 2,
          -1: 2,
          74: 1,
          50: 1,
          31: 1,
          93: 1,
          24: 1,
          82: 1}),
 3: Counter({41: 2,
          8: 2,
          76: 2,
          -1: 2,
          63: 1,
          58: 1,
          92: 1,
          90: 1,
          54: 1,
          70: 1,
          66: 1,
          98: 1}),
 4: Counter({30: 5,
          83: 4,
          49: 3,
          19: 2,
          28: 1,
          78: 1,
          4: 1,
          73: 1,
          13: 1,
          -1: 1}),
 5: Counter({21: 5, 74: 4, 5: 4, 52: 4, 83: 3, 0: 2, 32: 2, 95: 1}),
 6: Counter({44: 4, 78: 3, 61: 2, 70: 2, 57: 2, 2: 2, 31: 2, 13: 1}),
 7: Counter({48: 6,
 

In [ ]:
def get_coords(out_count, orient):
    stones = []
    if orient == 'north':
        for cx, stops in out_count.items():
            for rx, cnt in stops.items(): 
                for i in range(1, cnt + 1):
                    stones.append((rx + i, cx))
    elif orient == 'south':
        for cx, stops in out_count.items():
            for rx, cnt in stops.items():
                for i in range(1, cnt + 1):
                    stones.append((rx - i, cx))
    elif orient == 'west':
        for rx, stops in out_count.items():
            for cx, cnt in stops.items():
                for i in range(1, cnt + 1):
                    stones.append((rx, cx + i))  
    elif orient == 'east':
        for rx, stops in out_count.items():
            for cx, cnt in sorted(stops.items()): # sort for comparison later
                for i in range(1, cnt + 1):
                    stones.append((rx, cx - i))
    return stones
get_coords(a, 'north')

[(1, 0),
 (2, 0),
 (15, 0),
 (18, 0),
 (19, 0),
 (24, 0),
 (25, 0),
 (26, 0),
 (37, 0),
 (41, 0),
 (42, 0),
 (47, 0),
 (57, 0),
 (60, 0),
 (67, 0),
 (68, 0),
 (69, 0),
 (78, 0),
 (79, 0),
 (80, 0),
 (81, 0),
 (82, 0),
 (83, 0),
 (84, 0),
 (2, 1),
 (6, 1),
 (16, 1),
 (17, 1),
 (18, 1),
 (32, 1),
 (36, 1),
 (37, 1),
 (52, 1),
 (53, 1),
 (56, 1),
 (57, 1),
 (58, 1),
 (67, 1),
 (94, 1),
 (95, 1),
 (96, 1),
 (0, 2),
 (1, 2),
 (7, 2),
 (8, 2),
 (14, 2),
 (15, 2),
 (25, 2),
 (32, 2),
 (34, 2),
 (35, 2),
 (36, 2),
 (51, 2),
 (63, 2),
 (64, 2),
 (65, 2),
 (75, 2),
 (83, 2),
 (94, 2),
 (0, 3),
 (1, 3),
 (9, 3),
 (10, 3),
 (42, 3),
 (43, 3),
 (55, 3),
 (59, 3),
 (64, 3),
 (67, 3),
 (71, 3),
 (77, 3),
 (78, 3),
 (91, 3),
 (93, 3),
 (99, 3),
 (0, 4),
 (5, 4),
 (14, 4),
 (20, 4),
 (21, 4),
 (29, 4),
 (31, 4),
 (32, 4),
 (33, 4),
 (34, 4),
 (35, 4),
 (50, 4),
 (51, 4),
 (52, 4),
 (74, 4),
 (79, 4),
 (84, 4),
 (85, 4),
 (86, 4),
 (87, 4),
 (1, 5),
 (2, 5),
 (6, 5),
 (7, 5),
 (8, 5),
 (9, 5),
 (22, 5),

In [30]:
def one_cycle(stones):
    cycle = ['north', 'west', 'south', 'east']
    for o in cycle:
        stones = get_coords(tilt(stones, o), o)
    return tuple(stones)

In [ ]:
hist = {}
stones = stones_init
for i in range(1, 1000):
    stones = one_cycle(stones)
    if stones in hist:
        print(f'Match found! Identical results after {hist[stones]} and {i} cycles. Periodicity {i - hist[stones]} cycles')
        break
    hist[stones] = i

Match found! Identical results after 93 and 127 cycles. Periodicity 34 cycles


In [42]:
# with an initial spin-up time of 93 cycles and a periodicity of 34 cycles, the state after x cycles should be equivalent to the state after (x - 93) mod 34 + 93 cycles
x = 1000000000
equiv = (x - 93) % 34 + 93
equiv

126

In [45]:
final_stones = [s for s, id in hist.items() if id == equiv][0]

In [46]:
# tallying:
total = 0
for rx, cx in final_stones:
    tally = len(data) - rx
    total += tally
total

112452

That's the right answer! You are one gold star closer to restoring snow operations.

You have completed Day 14! You can [Share] this victory or [Return to Your Advent Calendar].

In [ ]:
# Puh glad there was no off-by-one-error, that would've been annoying to fix